# Santec TSL-570 Tunable Semiconductor Laser

This notebook demonstrates the QCoDeS driver for the Santec TSL-570 tunable laser.

**Key Features:**
- SCPI command set (Standard Commands for Programmable Instruments)
- SI units throughout (meters for wavelength, Hz for frequency, Watts for power)
- Ethernet/TCP-IP connection via `IPInstrument`
- Wavelength range: 1260-1680 nm (model dependent)
- Sweep modes: continuous and step-wise
- External trigger support

**Important:** Update the IP address and port to match your instrument before running.

## 1. Import and Connect

In [40]:
import time

from qcodes_contrib_drivers.drivers.Santec.Santec_TSL570 import SantecTSL570

# Connect to the instrument (update IP address and port for your setup)
laser = SantecTSL570("laser", "192.168.50.29", port=5000)

print(f"Connected to: {laser.get_idn()}")

Connected to: SANTEC TSL-570 (serial:21090055, firmware:0010.0007.0003) in 0.13s
Connected to: {'vendor': 'SANTEC', 'model': 'TSL-570', 'serial': '21090055', 'firmware': '0010.0007.0003'}


## 2. System Information

Query basic instrument information and status.

In [41]:
print("=== System Information ===")
print(f"Firmware version:  {laser.system_version()}")
print(f"Product code:      {laser.system_code()}")
print(f"Command set:       {laser.command_set_param()}")
print(f"Interlock status:  {laser.system_lock()}")
print(f"Error queue:       {laser.system_error()}")

=== System Information ===
Firmware version:  0010.0007.0003
Product code:      A-500630-P-F-AP-00-1
Command set:       SCPI
Interlock status:  False
Error queue:       {'code': -113, 'message': 'Undefined header'}


## 3. Basic Wavelength and Power Control

**Remember:** All wavelength values use SI units (meters).

In [42]:
# Query current settings
print("=== Current Settings ===")
print(f"Wavelength: {laser.wavelength() * 1e9:.4f} nm ({laser.wavelength():.10e} m)")
print(f"Frequency:  {laser.frequency() / 1e12:.6f} THz ({laser.frequency():.10e} Hz)")
print(f"Power:      {laser.power():.4f} mW")
print(f"Output:     {laser.output()}")
print(f"Shutter:    {laser.shutter()}")

=== Current Settings ===
Wavelength: 1552.0000 nm (1.5520000000e-06 m)
Frequency:  193.165240 THz (1.9316524000e+14 Hz)
Power:      1.0000 mW
Output:     False
Shutter:    True


In [43]:
# Set wavelength (in meters!)
target_wavelength_nm = 1550.0
laser.wavelength(target_wavelength_nm * 1e-9)
print(f"Wavelength set to: {laser.wavelength() * 1e9:.4f} nm")

# Alternative: Set using frequency
target_frequency_THz = 193.5
laser.frequency(target_frequency_THz * 1e12)
print(f"Frequency set to: {laser.frequency() / 1e12:.4f} THz")
print(f"Corresponding wavelength: {laser.wavelength() * 1e9:.4f} nm")

Wavelength set to: 1550.0000 nm
Frequency set to: 193.5000 THz
Corresponding wavelength: 1549.3150 nm


In [44]:
# Configure output power
laser.power(1.0)  # 1 mW
print(f"Power set to: {laser.power():.4f} mW")

# Enable automatic power control
laser.power_auto(True)
print(f"Auto power control: {laser.power_auto()}")

# Enable laser output
laser.output(True)
laser.shutter(True)
print(f"Output enabled: {laser.output()}")
print(f"Shutter open: {laser.shutter()}")

# Wait for stabilization and check actual power
time.sleep(0.5)
print(f"Actual power: {laser.power_actual():.4f} mW")

Power set to: 1.0000 mW
Auto power control: True
Output enabled: False
Shutter open: True
Actual power: 0.0000 mW


## 4. Fine Wavelength Control

In [45]:
# Fine-tuning offset (-100 to +100, unitless)
laser.wavelength_fine(25.0)
print(f"Fine tuning offset: {laser.wavelength_fine()}")

# Permanent wavelength offset (in meters)
laser.wavelength_offset(0.01e-9)  # 0.01 nm
print(f"Wavelength offset: {laser.wavelength_offset() * 1e9:.3f} nm")

# Disable fine-tuning
laser.disable_fine_tuning()
print("Fine-tuning disabled")

Fine tuning offset: 25.0
Wavelength offset: 0.010 nm
Fine-tuning disabled


## 5. Wavelength Sweep Configuration

Configure sweep parameters for continuous or step-wise wavelength scanning.

In [46]:
# Set sweep range using wavelength (in meters)
start_nm = 1530.0
stop_nm = 1570.0

laser.sweep_start_wavelength(start_nm * 1e-9)
laser.sweep_stop_wavelength(stop_nm * 1e-9)

print("=== Wavelength Sweep Configuration ===")
print(f"Start: {laser.sweep_start_wavelength() * 1e9:.2f} nm")
print(f"Stop:  {laser.sweep_stop_wavelength() * 1e9:.2f} nm")
print(f"Range: {(laser.sweep_stop_wavelength() - laser.sweep_start_wavelength()) * 1e9:.2f} nm")

=== Wavelength Sweep Configuration ===
Start: 1530.00 nm
Stop:  1570.00 nm
Range: 40.00 nm


In [47]:
# Alternative: Set sweep range using frequency
start_THz = 187.3
stop_THz = 196.1

laser.sweep_start_frequency(start_THz * 1e12)
laser.sweep_stop_frequency(stop_THz * 1e12)

print("=== Frequency Sweep Configuration ===")
print(f"Start: {laser.sweep_start_frequency() / 1e12:.2f} THz")
print(f"Stop:  {laser.sweep_stop_frequency() / 1e12:.2f} THz")

=== Frequency Sweep Configuration ===
Start: 187.30 THz
Stop:  196.10 THz


In [48]:
# Configure sweep mode and speed
# Modes: 0=step/one-way, 1=continuous/one-way, 2=step/two-way, 3=continuous/two-way
laser.sweep_mode(1)  # Continuous, one-way
laser.sweep_speed(10)  # 10 nm/s

# For step mode, configure step and dwell time
laser.sweep_step(0.1e-9)  # 0.1 nm step (in meters)
laser.sweep_dwell(0.05)  # 50 ms dwell time

# Set sweep cycles and delay
laser.sweep_cycles(1)  # Number of repetitions
laser.sweep_delay(0.5)  # Delay between sweeps

print("=== Sweep Parameters ===")
print(f"Mode:   {laser.sweep_mode()} (0=step/1-way, 1=cont/1-way, 2=step/2-way, 3=cont/2-way)")
print(f"Speed:  {laser.sweep_speed()} nm/s")
print(f"Step:   {laser.sweep_step() * 1e9:.3f} nm")
print(f"Dwell:  {laser.sweep_dwell()} s")
print(f"Cycles: {laser.sweep_cycles()}")
print(f"Delay:  {laser.sweep_delay()} s")

=== Sweep Parameters ===
Mode:   1 (0=step/1-way, 1=cont/1-way, 2=step/2-way, 3=cont/2-way)
Speed:  10.0 nm/s
Step:   0.100 nm
Dwell:  0.1 s
Cycles: 1
Delay:  0.5 s


## 6. Execute and Monitor Sweep

In [49]:
# Check initial state
print(f"Sweep state before start: {laser.sweep_state()}")

# Start a single sweep
laser.sweep_single()
time.sleep(0.2)

# Monitor sweep progress
print("\n=== Monitoring Sweep ===")
for i in range(5):
    state = laser.sweep_state()
    wl = laser.wavelength() * 1e9
    count = laser.sweep_count()
    print(f"[{i + 1}] State: {state:16s}  λ = {wl:7.3f} nm  Count: {count}")
    time.sleep(.1)
    if state == "STOPPED":
        break

# Stop sweep if still running
laser.sweep_stop()
print(f"\nSweep stopped. Final state: {laser.sweep_state()}")

Sweep state before start: STOPPED

=== Monitoring Sweep ===
[1] State: STOPPED           λ = 1549.315 nm  Count: 1

Sweep stopped. Final state: STOPPED


## 7. Trigger Configuration

Configure external trigger input and output for synchronized measurements.

In [50]:
# Configure trigger input
laser.trigger_input_external(True)
laser.trigger_input_polarity("RISE")
laser.trigger_input_standby(False)

print("=== Trigger Input ===")
print(f"External trigger: {laser.trigger_input_external()}")
print(f"Polarity:         {laser.trigger_input_polarity()}")
print(f"Standby mode:     {laser.trigger_input_standby()}")

=== Trigger Input ===
External trigger: True
Polarity:         RISE
Standby mode:     False


In [51]:
# Configure trigger output
laser.trigger_output_timing("STEP")  # Options: "NONE", "STOP", "START", "STEP"
laser.trigger_output_polarity("RISE")
laser.trigger_output_step(1e-12)  # 1 pm interval (in meters)
laser.trigger_output_setting("WAVELENGTH")  # or "TIME"

print("\n=== Trigger Output ===")
print(f"Timing:    {laser.trigger_output_timing()}")
print(f"Polarity:  {laser.trigger_output_polarity()}")
print(f"Step:      {laser.trigger_output_step() * 1e12:.3f} pm")
print(f"Mode:      {laser.trigger_output_setting()}")


=== Trigger Output ===
Timing:    STEP
Polarity:  RISE
Step:      1.000 pm
Mode:      WAVELENGTH


## 8. QCoDeS Snapshot

Capture the complete instrument state using QCoDeS snapshot functionality.

In [52]:


snapshot = laser.snapshot(update=True)

# Print a formatted version (truncated for readability)
print("=== Instrument Snapshot (selected parameters) ===")
params = snapshot['parameters']
for param_name in ['wavelength', 'frequency', 'power', 'output', 'sweep_mode', 'sweep_state']:
    if param_name in params:
        param = params[param_name]
        print(f"{param_name:20s}: {param.get('value', 'N/A')}")

# Uncomment to see full snapshot
# print(json.dumps(snapshot, indent=2, default=str))

[laser(SantecTSL570)] Snapshot: Could not update parameter: sweep_range_minimum
[laser(SantecTSL570)] Snapshot: Could not update parameter: sweep_range_maximum


=== Instrument Snapshot (selected parameters) ===
wavelength          : 1.549315e-06
frequency           : 193500000000000.0
power               : 1.0
output              : False
sweep_mode          : 1
sweep_state         : STOPPED


## 9. Cleanup and Shutdown

In [53]:
# Safe shutdown sequence
print("Shutting down laser...")

# Stop any running sweep
laser.sweep_stop()
print(f"Sweep stopped: {laser.sweep_state()}")

# Disable output
laser.shutter(False)
laser.output(False)
print(f"Output disabled: {laser.output()}")
print(f"Shutter closed: {laser.shutter()}")

# Close connection
laser.close()
print("Connection closed.")

Shutting down laser...
Sweep stopped: STOPPED
Output disabled: False
Shutter closed: True
Connection closed.
